In [86]:
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
import pickle
import threading
import sys
import json


In [3]:
# To help request from the website
base_url = 'https://www.federalreserve.gov'
cal_url = f'{base_url}/monetarypolicy/fomccalendars.htm'
history_url = f"{base_url}/monetarypolicy/fomc_historical_year.htm"
history_year = 2020
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/122.0.0.0 Safari/537.36'
}

In [10]:
def get_fomc_minutes_links(years):
    res_links = []
    response = requests.get(cal_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    links = soup.select('a[href*="/monetarypolicy/fomcminutes"]')

    for link in links:
        text = link.get_text(strip=True)
        href = link['href']
        if not href.startswith('http'):
            href = base_url + href
        for year in years:
            if str(year) in text or str(year) in href:
                res_links.append(href)

    # Deal with history years(2018,2019)
    for year in years:
        if year<history_year:
            curr_url = 'https://www.federalreserve.gov/monetarypolicy/fomchistorical'+ str(year) + '.htm'
            his_response = requests.get(curr_url, headers=headers)
            his_soup = BeautifulSoup(his_response.text, 'html.parser')
            his_links = his_soup.select('a[href*="/monetarypolicy/fomcminutes"]')

            for link in his_links:
                text = link.get_text(strip=True)
                href = link['href']
                if not href.startswith('http'):
                    href = base_url + href

                res_links.append(href)

    return res_links

In [19]:
def get_date_from_link(link):
    date = re.findall(r'\d+', link)[0]
    # date = "{}-{}-{}".format(date[:4], date[4:6], date[6:])
    return date

In [90]:
# To help test get_fomc_minutes_links()
'''
years = [2018,2019,2024,2025]
res = get_fomc_minutes_links(years)
print(len(res))
for link in res:
    print(link,get_date_from_link(link))
'''

'\nyears = [2018,2019,2024,2025]\nres = get_fomc_minutes_links(years)\nprint(len(res))\nfor link in res:\n    print(link,get_date_from_link(link))\n'

In [73]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text) 
    text = re.sub(r'\xa0', ' ', text) 
    text = re.sub(r'\s*Return to text\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\d+\]', '', text) 
    return text.strip()

In [87]:
# Get FOMC Meeting Minutes transcripts in 2018&2019, 2024&2025
def get_fomc_minutes_transcript(years, name):
    articles = []
    dates = []
    links = get_fomc_minutes_links(years)

    for link in links:
        minutes = requests.get(link, headers = headers)
        soup = BeautifulSoup(minutes.text, 'html.parser')
        main_content = soup.find('div', class_='col-xs-12 col-sm-8 col-md-9')

        article_div = soup.find('div', {'id': 'article'})
        paragraphs = []
        for tag in article_div.find_all(['p']):
            if tag.find('a') or tag.find('sup'):
                continue
            txt = tag.get_text(separator=' ', strip=True)
            if txt:
                paragraphs.append(clean_text(txt))
        article = '\n'.join(paragraphs)
        # paragraphs = main_content.findAll('p')
        #clean_text = "\n\n".join(p.get_text(strip = True) for p in paragraphs if p.get_text(strip=True))
        # article = clean_text.replace('\n', ' ').replace(',', ';').strip()
        data = {}
        data['date'] = get_date_from_link(link)
        data['content'] = article
        articles.append(data)
        # print(main_content)
    with open(name, 'w', encoding='utf-8') as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)
    return articles

In [89]:
years1 = [2018,2019]
years2 = [2024,2025]
c1 = get_fomc_minutes_transcript(years1,"FOMC_minutes_2018-2019.json")
c2 = get_fomc_minutes_transcript(years2,"FOMC_minutes_2024-2025.json")


In [ ]:
# Get FOMC Press Conference transcripts in 2018&2019,2024&2025
def get_fomc_press_conference_transcript():
    